# 06 — Error analysis (FP / FN / confusion)

Analyze false positives, false negatives, and multiclass confusions from trained artifacts.

**Prerequisite:** `models/trained_models/` and processed splits in `data/processed/`.

This notebook supports the results chapter — it does not retrain models.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("..").resolve()
report = json.loads((ROOT / "models/trained_models/training_report.json").read_text(encoding="utf-8"))
bin_test = report["binary"]["test"]

display(Markdown("## Binary detection error profile"))
rows = {
    "precision": bin_test.get("precision"),
    "recall": bin_test.get("recall"),
    "f1": bin_test.get("f1"),
    "fpr": bin_test.get("fpr"),
    "fnr": bin_test.get("fnr"),
    "specificity": bin_test.get("specificity"),
    "pr_auc": bin_test.get("pr_auc"),
    "roc_auc": bin_test.get("roc_auc"),
    "mcc": bin_test.get("mcc"),
}
display(pd.DataFrame([rows]).T.rename(columns={0: "value"}))

cm = np.array(bin_test["confusion_matrix"])
if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    display(Markdown(
        f"""### Counts\n"
- True negatives (benign→benign): **{tn}**\n"
- False positives (benign→attack): **{fp}**\n"
- False negatives (attack→benign): **{fn}**\n"
- True positives (attack→attack): **{tp}**\n"
\n"
**Interpretation for IDS:** FPs waste analyst time; FNs miss real attacks. Prefer high recall with controlled FPR.\n"
"""
    ))


In [ ]:
display(Markdown("## Multiclass confusion overview"))
mc = report["multiclass"]["test"]
classes = report["multiclass"]["classes"]
mcm = np.array(mc["confusion_matrix"])
cm_df = pd.DataFrame(mcm, index=classes, columns=classes)
display(Markdown("Rows = true label, columns = predicted"))
display(cm_df)

pairs = []
for i, true_cls in enumerate(classes):
    for j, pred_cls in enumerate(classes):
        if i == j:
            continue
        n = int(mcm[i, j])
        if n > 0:
            pairs.append((n, true_cls, pred_cls))
pairs.sort(reverse=True)
display(Markdown("### Most common confusions"))
display(pd.DataFrame(pairs[:15], columns=["count", "true", "predicted"]))
